In [2]:
import torch
from torch.utils.data import DataLoader
from seqeval.metrics import classification_report
from transformers import AutoTokenizer

from evaluate import evaluate_bert_bilstm_crf
from models.bert_bilstm_crf import BertBiLstmCrfNER
from utils.bert_bilstm_crf.data_utils import read_conll_2, NERDataset, build_collate_fn

In [3]:
MAX_LENGTH = 128
BATCH_SIZE = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_PATH = '../data/matscholar/test.txt'
MODEL_PATH = '../models/bert_bilstm_crf.pt'

In [4]:
# 读取模型保存文件
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

word2idx = checkpoint['word2idx']
tag2idx = checkpoint['tag2idx']
idx2tag = {v: k for k, v in tag2idx.items()}

MODEL_NAME = checkpoint.get('model_name', 'bert-base-cased')
WORD_EMBEDDING_DIM = checkpoint.get('word_embedding_dim', 128)
LSTM_HIDDEN_SIZE = checkpoint.get('lstm_hidden_size', 256)
DROPOUT = checkpoint.get('dropout', 0.25)

# 读取测试集
test_sentence, test_tags = read_conll_2(TEST_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

test_dataset = NERDataset(test_sentence, test_tags)

collate_fn = build_collate_fn(
    tokenizer=tokenizer,
    label2id=tag2idx,
    word2idx=word2idx,
    max_length=MAX_LENGTH
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# 建立模型
model = BertBiLstmCrfNER(
    model_name=MODEL_NAME,
    num_labels=len(tag2idx),
    word_vocab_size=len(word2idx),
    word_embedding_dim=WORD_EMBEDDING_DIM,
    lstm_hidden_size=LSTM_HIDDEN_SIZE,
    dropout=DROPOUT,
    word_pad_idx=word2idx['<PAD>'],
    id2label=idx2tag,
    label2id=tag2idx
).to(DEVICE)

# 加载参数
model.load_state_dict(checkpoint['model'])

# 测试评估
test_loss, test_p, test_r, test_f1, y_true, y_pred = evaluate_bert_bilstm_crf(
    model, test_loader, idx2tag, DEVICE
)

print("\n========== Test Result ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Precision: {test_p:.4f}")
print(f"Test Recall: {test_r:.4f}")
print(f"Test F1: {test_f1:.4f}")

print("\n========== Detailed Report ==========")
print(classification_report(y_true, y_pred, digits=4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



========== Test Result ==========
Test Loss: 7.0289
Test Precision: 0.8235
Test Recall: 0.8537
Test F1: 0.8384

========== Detailed Report ==========
              precision    recall  f1-score   support

         APL     0.6923    0.6702    0.6811        94
         CMT     0.8410    0.8913    0.8654       184
         DSC     0.8864    0.8930    0.8897       402
         MAT     0.8438    0.9054    0.8735       698
         PRO     0.7810    0.7923    0.7866       621
         SMT     0.7908    0.8514    0.8200       222
         SPL     0.8919    0.7857    0.8354        42

   micro avg     0.8235    0.8537    0.8384      2263
   macro avg     0.8182    0.8270    0.8217      2263
weighted avg     0.8233    0.8537    0.8379      2263

